In [36]:
# 데이터 생성
import pandas as pd
dataset = pd.read_csv('../../data/csv/movie_reviews2.csv')

In [37]:
# 독립, 종속
X = dataset.iloc[ :, -1]
y = dataset.iloc[ :, 0]

In [38]:
# 형태소 분석
from konlpy.tag import Okt
okt = Okt()

def tokenized_korean(text_list):
  return ["".join(okt.morphs(text, stem=True)) for text in text_list]

morphs_korean = tokenized_korean(X)

In [39]:
# 벡터화
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode='int',
	output_sequence_length = 10
)

vectorize_layer.adapt(morphs_korean)

In [40]:
# 파이프라인
import tensorflow as tf
train_ds = tf.data.Dataset.from_tensor_slices((morphs_korean,y)).batch(16)


In [41]:
# 모델 설계
def build_positional_encoding():
  
  # 입력층
  inputs = layers.Input((1,), dtype=tf.string)
  
  # 벡터화
  x = vectorize_layer(inputs)
  
  # 임베딩 : 단어를
  vocab_size = vectorize_layer.vocabulary_size()
  word_embeding = layers.Embedding(input_dim=vocab_size, output_dim=64)(x)
  
  # 포지셔널 인코딩
  # 위치 정보 추가 : 0~9 번호를 생성
  positions = tf.range(start=0, limit=10, delta=1)
  # 임베딩 : 위치를
  pos_embeding = layers.Embedding(input_dim=10, output_dim=64)(positions)
  
  # 의미 + 위치
  x = word_embeding + pos_embeding
  
  # 멀티헤드 어텐션
  attention_output = layers.MultiHeadAttention(num_heads=2, key_dim=64)(x,x)
  
  # 잔차 계산 및 정규화
  x = layers.Add()([x,attention_output])
  x = layers.LayerNormalization()(x)
  
  # ffn
  ffn = layers.Dense(64, activation='relu')(x)
  x = layers.Add()([x, ffn])
  x = layers.LayerNormalization()(x)
  
  # 압축
  x = layers.GlobalAveragePooling1D()(x)
  
  # 출력층
  outputs = layers.Dense(1, activation='sigmoid')(x)
  
  return models.Model(inputs=inputs, outputs=outputs)

model = build_positional_encoding()

In [42]:
# 모델 설정
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [43]:
# 모델 학습
model.fit(train_ds, epochs=50)

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.5000 - loss: 0.8079
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5000 - loss: 0.7112 
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5000 - loss: 0.7284
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6000 - loss: 0.6591 
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5000 - loss: 0.6728
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7000 - loss: 0.6321
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6667 - loss: 0.6182
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5333 - loss: 0.6087
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9333 - loss: 0.5780
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8667 - loss: 0.5647
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9333 - loss: 0.5463
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 1.0000 - loss: 0.5196
E

In [44]:
#예측

# 테스트
test_text = ["영화가 너무 재미있어서 하나도 안 지루해요"]

# 형태소별로 문장을 수정
test_morphs = tokenized_korean(test_text)

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
predictions = model.predict(test_input)
predictions_size = len(predictions)
for i in range(predictions_size):
  p = predictions[i][0] * 100
  print(f'{test_text[i]}:{'긍정' if p > 50 else '부정'}({p:.2f}%)')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
영화가 너무 재미있어서 하나도 안 지루해요:긍정(99.96%)
